<a href="https://colab.research.google.com/github/SestamibiTechLab/Transcriber/blob/main/Transcribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Podcast Transcriber - Colab Backend
Run this notebook to start a transcription server on Google Colab with GPU support.

1. Add your ngrok authtoken in the cell below
2. Run all cells
3. Copy the public URL to your frontend

In [5]:
# Install dependencies
!pip install -q flask flask-cors pyngrok yt-dlp openai-whisper
!apt-get update > /dev/null 2>&1 && apt-get install -y ffmpeg > /dev/null 2>&1
print("Dependencies installed!")

Dependencies installed!


In [6]:
import os
from pyngrok import ngrok

# PASTE YOUR NGROKA UTHTOKEN HERE
NGROK_AUTHTOKEN = "3DrJOl4iNWcySSkKdeJvYjcY99z_2qNHLJFZqrxY6JbnD18PD"

if NGROK_AUTHTOKEN == "YOUR_NGROK_AUTHTOKEN_HERE":
    raise ValueError("\n⚠️  Please paste your ngrok authtoken in the cell above!\nGet it from: https://dashboard.ngrok.com/auth/your-authtoken")

ngrok.set_auth_token(NGROK_AUTHTOKEN)
print("✅ ngrok configured")

✅ ngrok configured


In [7]:
import os
import whisper
import tempfile
import yt_dlp
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

app = Flask(__name__)
CORS(app)

# Load model once (GPU will be used automatically)
print("Loading Whisper model (base)...")
model = whisper.load_model("base", device="cuda")
print("✅ Model loaded on GPU")

@app.route("/health")
def health():
    return jsonify({"status": "ok", "model": "base", "device": "cuda"})

@app.route("/transcribe", methods=["POST"])
def transcribe():
    data = request.get_json()
    url = (data or {}).get("url", "").strip()

    if not url:
        return jsonify({"error": "No URL provided"}), 400

    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            audio_path = os.path.join(tmpdir, "audio.mp3")

            # yt-dlp options
            ydl_opts = {
                "format": "bestaudio/best",
                "outtmpl": audio_path,
                "postprocessors": [{
                    "key": "FFmpegExtractAudio",
                    "preferredcodec": "mp3",
                    "preferredquality": "128",
                }],
                "quiet": True,
                "no_warnings": True,
                # Allow up to 2 hours (more generous with GPU)
                "match_filter": yt_dlp.utils.match_filter_func("duration < 7200"),
            }

            print(f"Downloading: {url}")
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])

            # Handle yt-dlp extension appending
            if not os.path.exists(audio_path):
                audio_path = audio_path + ".mp3"

            if not os.path.exists(audio_path):
                return jsonify({"error": "Could not download audio from that URL"}), 400

            file_size = os.path.getsize(audio_path) / (1024 * 1024)  # MB
            print(f"Downloaded: {file_size:.1f}MB")

            # Transcribe with GPU
            print(f"Transcribing...")
            result = model.transcribe(audio_path, language=data.get("language") if data.get("language") != "auto-detect" else None)

            segments = [
                {"start": round(s["start"], 1), "end": round(s["end"], 1), "text": s["text"].strip()}
                for s in result.get("segments", [])
            ]

            return jsonify({
                "text": result["text"].strip(),
                "language": result.get("language", "unknown"),
                "segments": segments,
            })

    except yt_dlp.utils.DownloadError as e:
        print(f"Download error: {str(e)[:200]}")
        return jsonify({"error": f"Download failed: {str(e)[:200]}"}), 400
    except Exception as e:
        print(f"Error: {str(e)[:300]}")
        return jsonify({"error": f"Transcription failed: {str(e)[:300]}"}), 500

print("✅ Flask app created")

Loading Whisper model (base)...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 163MiB/s]


✅ Model loaded on GPU
✅ Flask app created


In [ ]:
from pyngrok import ngrok
import time

# Start Flask in background
def run_flask():
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)  # Give Flask time to start

# Expose via ngrok
print("🚀 Starting ngrok tunnel...")
public_url = ngrok.connect(5000, "http")
print(f"\n✅ Server is live!")
print(f"\n📍 Public URL: {public_url}")
print(f"\n📋 Copy this URL to your frontend's API URL field")
print(f"\n✨ Server will run as long as this cell is executing")
print(f"\n(Don't close this notebook - it will stop the server)")

# Keep the server running
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\n⏹️  Stopping server...")
    ngrok.disconnect(public_url)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


🚀 Starting ngrok tunnel...

✅ Server is live!

📍 Public URL: NgrokTunnel: "https://multiple-dinghy-darkening.ngrok-free.dev" -> "http://localhost:5000"

📋 Copy this URL to your frontend's API URL field

✨ Server will run as long as this cell is executing

(Don't close this notebook - it will stop the server)


INFO:werkzeug:127.0.0.1 - - [17/May/2026 21:09:12] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 21:09:15] "GET /health HTTP/1.1" 200 -
